# 8 · Unsteady problems — the double-glazing flow

Real processes **evolve in time**. The recipe is almost always the same: discretise **space**
as before (a weak form, an FE space), then march **time** in small steps. We meet the simplest
and most robust marcher — **implicit Euler** — on the classic **double-glazing** benchmark: a
temperature that is **diffused** and **carried by a recirculating wind**.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from netgen.geom2d import SplineGeometry
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt

## 1. The double-glazing problem

On the square $\Omega=(-1,1)^2$ a temperature $u(t,\mathbf x)$ is **heated** on the right wall
($u=1$), held **cold** on the left ($u=0$), and the top & bottom are **insulated** (no heat
flux — a natural/Neumann boundary, so we simply leave them out of the Dirichlet set). A fixed
**recirculating wind** $\mathbf b$ stirs it:
$$ \partial_t u \;+\; \mathbf b\!\cdot\!\nabla u \;-\; \varepsilon\,\Delta u \;=\; 0,
   \qquad \mathbf b(x,y)=\bigl(2y(1-x^2),\,-2x(1-y^2)\bigr). $$
The wind runs **clockwise** in a single cell; with small diffusion $\varepsilon$ the heat is
**carried around** before it spreads.

In [ ]:
geo = SplineGeometry()
geo.AddRectangle((-1, -1), (1, 1), bcs=["bottom", "right", "top", "left"])
mesh = Mesh(geo.GenerateMesh(maxh=0.05))
wind = CF((2*y*(1-x*x), -2*x*(1-y*y)))                 # the recirculating draught
Draw(wind, mesh, "wind", vectors={"grid_size": 24})

## 2. Space, then time — implicit Euler

**Space** is the usual weak form on an `H1` space (Dirichlet on the hot/cold walls only). We
assemble two operators: the **mass** matrix $M$ (from $\int u\,v$) and the **stiffness**
$A$ (diffusion **+** convection, $\int \varepsilon\nabla u\!\cdot\!\nabla v + (\mathbf b\!\cdot\!
\nabla u)\,v$). **Time** is then *implicit Euler*: from $\partial_t u + Au = 0$,
$$ \frac{u^{n+1}-u^n}{\Delta t} + A\,u^{n+1} = 0 \;\Longrightarrow\;
   \underbrace{(M+\Delta t\,A)}_{M^{*}}\,u^{n+1} = M\,u^n . $$
$M^{*}$ is **constant**, so we **factorise it once** and only back-substitute each step.

In [ ]:
eps, dt = 0.05, 0.01
fes = H1(mesh, order=2, dirichlet="right|left")        # only the hot/cold walls are Dirichlet
u, v = fes.TnT()
M = BilinearForm(u*v*dx).Assemble()
A = BilinearForm(eps*grad(u)*grad(v)*dx + (wind*grad(u))*v*dx).Assemble()
mstar = M.mat.CreateMatrix()
mstar.AsVector().data = M.mat.AsVector() + dt*A.mat.AsVector()   # M* = M + dt·A, assembled once
inv = mstar.Inverse(fes.FreeDofs())                    # factor once, reuse every step

gfu = GridFunction(fes)
gfu.Set(CF(1), definedon=mesh.Boundaries("right"))     # hot right wall = 1, everything else 0
Draw(gfu, mesh, "temperature", min=0, max=1, autoscale=False)

## 3. Step in time — watch the heat go around

One step is a single back-substitution: form the right-hand side $M u^n$ (equivalently the
residual update below, which keeps the Dirichlet walls fixed), solve, repeat. We collect a few
frames to animate the heat being **dragged down the hot wall and swept around** the cell.

In [ ]:
tend, res = 2.5, gfu.vec.CreateVector()
nsteps = int(tend/dt + 0.5)
gx = gy = np.linspace(-1, 1, 90)
mips = [mesh(float(xx), float(yy)) for yy in gy for xx in gx]    # sample points, found once
frames = []
with TaskManager():
    for step in range(1, nsteps+1):
        res.data = -dt * A.mat * gfu.vec               # implicit-Euler residual (Dirichlet kept)
        gfu.vec.data += inv * res
        if step % 12 == 0:
            frames.append(np.array([gfu(mp) for mp in mips]).reshape(len(gy), len(gx)))
print(f"stepped to t={tend}:  temperature in [{min(gfu.vec):.2f}, {max(gfu.vec):.2f}],  {len(frames)} frames")

The animation (press play): the warm front leaves the right wall, is **carried down and to the
left** by the draught, and only slowly diffuses — the signature of a **convection-dominated**
transient.

In [ ]:
from matplotlib import animation
from IPython.display import HTML
fig, ax = plt.subplots(figsize=(4.6, 4.2))
def draw(i):
    ax.clear()
    ax.contourf(gx, gy, frames[i], levels=np.linspace(0, 1, 21), cmap="hot")
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"t = {12*(i+1)*dt:.2f}")
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=120)
plt.close(fig)
HTML(anim.to_jshtml())

## 4. Towards the steady state

Marched long enough, the transient settles to a **steady** recirculating temperature — the same
answer the stationary problem $\mathbf b\!\cdot\!\nabla u-\varepsilon\Delta u=0$ would give
directly. Implicit Euler is **unconditionally stable**, so we may take large steps to *reach*
that state cheaply; for *accuracy in time* one uses smaller steps or a higher-order scheme
(e.g. **Crank–Nicolson**, $\tfrac12$-implicit).

In [ ]:
Draw(gfu, mesh, "steady temperature", min=0, max=1, autoscale=False)

**Next:** the wind here was fixed and the equation **linear**. When the operator depends on the
solution itself, we step into **nonlinear** problems (unit 9).

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("07-saddle-point", "7 · Mixed problems — the saddle point 🐎")
    _next = ("09-nonlinear-allencahn", "9 · Nonlinear problems — Allen–Cahn & Newton")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))